# 🚀 Google Colab 极速上手与算力基准测试

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuler/explore-colab/blob/main/notebooks/01_colab_quickstart_benchmark.ipynb)

> **工作流说明**：本 Notebook 属于「GitHub 双向联动」模式。
> 点击上方徽章可直接在 Google Colab 中打开此文件；在云端修改后，通过菜单「文件」->「在 GitHub 中保存副本」即可直接将代码提交回 GitHub！


## 1. 硬件环境与系统诊断

检测当前分配的云端虚机硬件配置（CPU、RAM、GPU、CUDA 等）。

In [ ]:
import os
import platform
import psutil
import torch

print('=' * 50)
print(f'🖥️ 操作系统: {platform.system()} {platform.release()}')
print(f'🐍 Python 版本: {platform.python_version()}')
print(f'⚙️ CPU 核心数: {os.cpu_count()}')
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f'💾 系统总内存: {ram_gb:.2f} GB')
print('=' * 50)

gpu_available = torch.cuda.is_available()
print(f'🔥 PyTorch CUDA 可用状态: {gpu_available}')

if gpu_available:
    device_name = torch.cuda.get_device_name(0)
    device_count = torch.cuda.device_count()
    total_mem = torch.cuda.get_device_properties(0).total_mem / (1024 ** 3)
    print(f'🎮 检测到 GPU: {device_name} (共 {device_count} 块)')
    print(f'🧠 显存大小: {total_mem:.2f} GB')
    print(f'⚡ CUDA 版本: {torch.version.cuda}')
else:
    print('⚠️ 当前为 CPU 运行时！')
    print('💡 提示：点击右上角 Connect（连接）旁的下拉菜单，选择 Change runtime type（更改运行时类型），在 Hardware accelerator（硬件加速器）中可切换为 GPU (T4/L4/A100) 或 TPU。')


In [ ]:
# 终端命令快速诊断
!nvidia-smi 2>/dev/null || echo '(当前未挂载 NVIDIA GPU 运行时)'


## 2. 算力基准对比：CPU vs GPU 矩阵乘法

通过生成大规模稠密矩阵（如 5000x5000 浮点张量），对比 CPU 与 GPU 的矩阵乘法速度。CPU 侧是一次 `torch.matmul`（默认走多线程），没有单独做单核对比。

In [ ]:
import time

MATRIX_SIZE = 5000
print(f'正在准备测试数据：尺寸为 {MATRIX_SIZE}x{MATRIX_SIZE} 浮点张量...')

# 1. CPU 测试
cpu_a = torch.randn(MATRIX_SIZE, MATRIX_SIZE, dtype=torch.float32)
cpu_b = torch.randn(MATRIX_SIZE, MATRIX_SIZE, dtype=torch.float32)

print('⏳ 开始 CPU 矩阵乘法 (torch.matmul)...')
start_cpu = time.perf_counter()
cpu_c = torch.matmul(cpu_a, cpu_b)
cpu_duration = time.perf_counter() - start_cpu
print(f'⏱️ CPU 耗时: {cpu_duration:.4f} 秒')

# 2. GPU 测试 (若可用)
if torch.cuda.is_available():
    gpu_a = torch.randn(MATRIX_SIZE, MATRIX_SIZE, device='cuda', dtype=torch.float32)
    gpu_b = torch.randn(MATRIX_SIZE, MATRIX_SIZE, device='cuda', dtype=torch.float32)
    # 预热
    _ = torch.matmul(gpu_a[:100, :100], gpu_b[:100, :100])
    torch.cuda.synchronize()

    print('⚡ 开始 GPU 矩阵乘法 (CUDA)...')
    start_gpu = time.perf_counter()
    gpu_c = torch.matmul(gpu_a, gpu_b)
    torch.cuda.synchronize()
    gpu_duration = time.perf_counter() - start_gpu
    speedup = cpu_duration / gpu_duration
    print(f'🔥 GPU 耗时: {gpu_duration:.4f} 秒')
    print(f'🚀 算力加速比: {speedup:.2f}x 倍！')
else:
    print('ℹ️ 当前未分配 GPU。请点击右上角 Connect 下拉菜单中的 Change runtime type，将 Hardware accelerator 切换为 GPU 后再运行。')


## 3. 云端存储与环境交互

Colab 虚拟机是临时的，关掉运行时后磁盘内容会消失。需要留下文件时，可以挂载 Google Drive。

In [ ]:
# 如需挂载 Google Drive，可取消下方注释执行
# from google.colab import drive
# drive.mount('/content/drive')

# 检查当前运行时文件系统
print('当前工作目录:', os.getcwd())
print('文件列表:', os.listdir('.'))


## 4. 同步回本地闭环

- 点击 Colab 菜单 **「文件」** -> **「在 GitHub 中保存副本」** (Save a copy in GitHub)。
- 选择仓库 `yuler/explore-colab`，分支 `main`，文件路径保持 `notebooks/01_colab_quickstart_benchmark.ipynb`，再填入提交信息。路径如果被改成仓库根目录，本地 `git pull` 会对不上这份笔记本。
- 在本地执行 `git pull`。拉回来的是这份笔记本本身，包括已经跑过的单元格输出。虚拟机里另外新建的文件不会一起回来。
